In [0]:
import logging
from pyspark.sql import functions as F
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("main")

In [0]:
%run ./../../common/utilities

In [0]:
dbutils.widgets.text("catalog", "abcgroup", "Catalog")
dbutils.widgets.text("table", "crm_sales_details", "Table")

In [0]:
catalog = dbutils.widgets.get("catalog")
table = dbutils.widgets.get("table")

In [0]:
df = (
    spark.table(f"{catalog}.{bronze_schema}.{table}")
)
display(df.limit(5))


In [0]:
df = df.select(
    [
        F.trim(F.col(field.name)).alias(field.name)
        if isinstance(field.dataType, StringType)
        else F.col(field.name)
        for field in df.schema.fields
    ]
)

In [0]:
display(df.limit(5))

In [0]:
date_cols = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]

for col_name in date_cols:
    df = df.withColumn(
        col_name,
        F.when(
            (F.col(col_name) == 0) | (F.length(F.col(col_name).cast("string")) != 8),
            None
        ).otherwise(F.to_date(F.col(col_name).cast("string"), "yyyyMMdd"))
    )

display(df.limit(5))

In [0]:
df = df.withColumn(
    "sls_price",
    F.coalesce(
        F.when(
            (F.col("sls_price").isNull()) | (F.col("sls_price") <= 0),
            F.when(F.col("sls_quantity") != 0, F.col("sls_sales") / F.col("sls_quantity"))
        ),
        F.col("sls_price")
    )
)

In [0]:
COL_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in COL_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df_flagged = (
    df.withColumn(
        "is_valid",
        F.when(
            F.col("order_number").isNotNull() &
            F.col("sales_amount").isNotNull() &
            F.col("quantity").isNotNull() &
            F.col("price").isNotNull(),
            1
        ).otherwise(0)
    )
)

In [0]:
valid_df = (
    df_flagged
    .filter(F.col("is_valid") == 1)
    .drop("is_valid")
)

invalid_df = (
    df_flagged
    .filter(F.col("is_valid") == 0)
    .drop("is_valid")
)

In [0]:
invalid_df = (
    invalid_df
    .withColumn("error_reason", F.lit("NULL in critical columns"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

In [0]:
invalid_count = invalid_df.count()
if invalid_count > 0:
    logger.warning(f"{invalid_count} invalid records moved to quarantine")

    (
        invalid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{catalog}.{silver_schema}.{table}_quarantine")
    )

In [0]:
display(valid_df.limit(5))

In [0]:
target_table = f"{catalog}.{silver_schema}.crm_sales"
valid_df.write.mode("overwrite").format("delta").saveAsTable(target_table)

logger.info("Saved table into %s in Delta format.", target_table)

In [0]:
display(valid_df.limit(5))

In [0]:
display(spark.sql(f"""
    SELECT *
    FROM {catalog}.{silver_schema}.crm_sales
    LIMIT 5
"""))